In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
import re
import nltk
nltk.download('punkt')

# Device selection: use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    # clear cache and enable cuDNN benchmark for potential speedups
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True

In [ ]:
# TRAIN_DATA_ROOT = "../data/target_4_December_release/EN/raw-documents"
# TRAIN_DATA_ANNOTATIONS = "../data/target_4_December_release/EN/subtask-1-annotations.txt"
# VAL_DATA_ROOT = "../data/cleaned_dev_10_january_2025/EN/subtask-1-documents"
# VAL_DATA_ANNOTATIONS = "../data/cleaned_dev_10_january_2025/EN/subtask-1-annotations.txt"
# TEST_DATA_ROOT = "../data/testdata_ST12/EN/subtask-1-documents"
# TEST_DATA_MENTIONS = "../data/testdata_ST12/EN/subtask-1-entity-mentions.txt"
# TEST_DATA_RESULTS = "../data/results"
# TAXONOMY_FILE = "../data/taxonomy.json"
# TEST_DATA_ROOT = VAL_DATA_ROOT
# TEST_DATA_MENTIONS = VAL_DATA_ANNOTATIONS

DATA_ROOT = "../data"
TRAIN_DATA_PARENT = os.path.join(DATA_ROOT, "target_4_December_release")
VAL_DATA_PARENT = os.path.join(DATA_ROOT, "cleaned_dev_10_january_2025")
TEST_DATA_PARENT = os.path.join(DATA_ROOT, "testdata_ST12")
TAXONOMY_FILE = os.path.join(DATA_ROOT, "taxonomy.json")
TEST_DATA_RESULTS = os.path.join(DATA_ROOT, "results")
TEST_DATA_PARENT = VAL_DATA_PARENT

In [ ]:
# === Detect all available language folders ===
available_languages = [d for d in os.listdir(TRAIN_DATA_PARENT) if os.path.isdir(os.path.join(TRAIN_DATA_PARENT, d))]
print("Detected languages:", available_languages)

with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

label_data = []

for category in taxonomy:
    for subtype in category["subtypes"]:
        label_data.append({
            "main_category": category["name"],
            "subtype": subtype["name"],
            "description": subtype["description"],
            "example": subtype["example"]
        })

df = pd.DataFrame(label_data)
df.head()

In [ ]:
main_categories = [cat["name"] for cat in taxonomy]
label2id = {label: i for i, label in enumerate(main_categories)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
# === Dataset preparation ===
# === Dataset loader ===
def load_annotations(annotation_path, docs_root, labeled=True):
    data = []
    with open(annotation_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if labeled:
                # if len(parts) < 5:
                #     continue
                doc_id, mention, start, end, *labels = parts
                label = labels[0]
            else:
                # if len(parts) < 4:
                #     continue
                doc_id, mention, start, end = parts
                label = None

            start, end = int(start), int(end)
            text_path = os.path.join(docs_root, doc_id)
            # if not os.path.exists(text_path):
            #     continue

            with open(text_path, "r", encoding="utf-8") as doc_file:
                text = doc_file.read()

            entry = {
                "doc_id": doc_id,
                "text": text,
                "mention": mention,
                "start": start,
                "end": end,
            }
            if labeled:
                entry["label"] = label
            data.append(entry)
    return pd.DataFrame(data)

import nltk
nltk.download("punkt")

# --------------------------------------------------------------------
# Utility: given text, produce (span_start, span_end, substring)
# for paragraphs or sentences, maintaining global offsets.
# --------------------------------------------------------------------
def split_paragraphs_with_offsets(text):
    parts = []
    offset = 0
    for raw in text.split("\n"):
        p = raw.strip()
        if not p:
            offset += len(raw) + 1  # still move offset
            continue
        start = text.index(raw, offset)
        end = start + len(raw) - 1
        parts.append((start, end, raw))
        offset = end + 1
    return parts


def split_sentences_with_offsets(text):
    parts = []
    sentences = nltk.sent_tokenize(text)

    search_offset = 0
    for s in sentences:
        idx = text.find(s, search_offset)
        if idx == -1:
            idx = text.index(s)  # fallback
        parts.append((idx, idx + len(s), s))
        search_offset = idx + len(s)
    return parts


# --------------------------------------------------------------------
# Main expander: document → paragraphs or sentences
# This enforces correct mention span logic.
# --------------------------------------------------------------------
def expand_to_smaller_units(df, mode):
    new_rows = []

    for _, row in df.iterrows():
        text = row["text"]
        mention = row["mention"]
        orig_start = row["start"]
        orig_end = row["end"]

        # ----------------------------------------------------------------
        # choose unit splitter
        # ----------------------------------------------------------------
        if mode == "paragraph":
            units = split_paragraphs_with_offsets(text)
        elif mode == "sentence":
            units = split_sentences_with_offsets(text)
        else:
            raise ValueError("Unsupported mode.")

        # ----------------------------------------------------------------
        # Only keep the chunk that actually contains the original mention
        # ----------------------------------------------------------------
        for unit_start, unit_end, unit_text in units:

            # Does this unit include the absolute mention span?
            if not (unit_start <= orig_start < unit_end):
                continue

            # ----------------------------------------------------------------
            # Recalculate local start/end inside this chunk
            # ----------------------------------------------------------------
            mention_length = orig_end - orig_start + 1

            local_start = orig_start - unit_start
            local_end   = local_start + mention_length - 1


            # Additional safety check:
            if unit_text[local_start:local_end] != mention:
                # fallback: search within the chunk for exact location
                # This handles rare tokenization boundary issues
                idx = unit_text.find(mention)
                if idx == -1:
                    continue
                local_start = idx
                local_end = idx + len(mention) - 1

            # Build new row
            new_row = row.copy()
            new_row["text"] = unit_text

            # local offsets for the model
            new_row["start"] = local_start
            new_row["end"] = local_end

            # store ORIGINAL global offsets (critical!)
            new_row["orig_start"] = orig_start
            new_row["orig_end"] = orig_end

            new_rows.append(new_row)


    return pd.DataFrame(new_rows)


# === Combine data from all languages ===
def load_multilingual_data_by_mode(mode="document", labeled=True):

    assert mode in ["document", "paragraph", "sentence"]

    all_train, all_val, all_test = [], [], []

    for lang in available_languages:
        print(f"\n📘 Loading {mode}-level data for language: {lang}")

        train_root = os.path.join(TRAIN_DATA_PARENT, lang, "raw-documents")
        train_ann  = os.path.join(TRAIN_DATA_PARENT, lang, "subtask-1-annotations.txt")

        val_root = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-documents")
        val_ann  = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-annotations.txt")

        test_root = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-documents")
        test_ann  = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-annotations.txt")

        if not (os.path.exists(train_ann) and os.path.exists(val_ann)):
            print(f"⚠️ Skipping {lang}: missing annotation files")
            continue

        # ======================
        # Load document-level data
        # ======================
        train_df_doc = load_annotations(train_ann, train_root, labeled=labeled)
        val_df_doc   = load_annotations(val_ann, val_root, labeled=labeled)
        test_df_doc  = load_annotations(test_ann, test_root, labeled=labeled) if os.path.exists(test_ann) else pd.DataFrame()

        # ======================
        # Transform based on mode
        # ======================
        if mode == "document":
            train_df, val_df, test_df = train_df_doc, val_df_doc, test_df_doc

        else:
            train_df = expand_to_smaller_units(train_df_doc, mode)
            val_df   = expand_to_smaller_units(val_df_doc, mode)
            test_df  = expand_to_smaller_units(test_df_doc, mode) if not test_df_doc.empty else pd.DataFrame()

        # append
        all_train.append(train_df)
        all_val.append(val_df)
        all_test.append(test_df)

    return (
        pd.concat(all_train, ignore_index=True),
        pd.concat(all_val, ignore_index=True),
        pd.concat(all_test, ignore_index=True)
    )

In [ ]:
# Load the data
train_df_full, val_df, test_df = load_multilingual_data_by_mode(mode="paragraph", labeled=True)

print(f"\n✅ Loaded dataset sizes: Train={len(train_df_full)}, Val={len(val_df)}, Test={len(test_df)}")

# Split the training data into new train and validation sets
train_df, new_val_df = train_test_split(train_df_full, test_size=0.2, random_state=42, stratify=train_df_full['label'])

# Use new validation set and keep using original validation set as test set
val_df = new_val_df
print(f"\n✅ Final dataset sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Print class distribution
print("\n=== Label distribution ===")
print("Training set:")
print(train_df['label'].value_counts().to_string())
print("\nValidation set:")
print(val_df['label'].value_counts().to_string())
print("\nTest set:")
print(test_df['label'].value_counts().to_string())

train_df.head()

In [ ]:
# === Dataset class ===
class EntityFramingDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len=256, labeled=True):
        self.df = df
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len
        self.labeled = labeled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]
        mention = row["mention"]
        marked_text = text.replace(mention, f"[ENTITY] {mention} [/ENTITY]")

        inputs = self.tokenizer(
            marked_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in inputs.items()}

        if self.labeled:
            label = self.label2id[row["label"]]
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


In [ ]:
# === Metrics ===
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# === Tokenizer and Model ===
model_name = "xlm-roberta-base" #"bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = EntityFramingDataset(train_df, tokenizer, label2id, labeled=True)
val_dataset = EntityFramingDataset(val_df, tokenizer, label2id, labeled=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Freeze BERT layers except for the last transformer block and classifier
# Generally, we keep the last transformer block (typically layers 10-12) and classifier trainable
for name, param in model.named_parameters():
    # Freeze everything except:
    # 1. The pooler layer
    # 2. The classifier layer
    # 3. The last encoder layer (layer 11 in BERT-base which has 12 layers)
    if not any(x in name for x in ['pooler', 'classifier', 'encoder.layer.11']):
        param.requires_grad = False

# Print trainable parameters summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Percentage of trainable parameters: {100 * trainable_params / total_params:.2f}%')

model.to(device)

# === Training ===
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


# === Evaluation ===
results = trainer.evaluate()
print("\n=== Validation Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# === Save trained model ===
model.save_pretrained("./entity_framing_baseline")
tokenizer.save_pretrained("./entity_framing_baseline")

# === Prediction on Test Set ===
print("\n=== Generating predictions on test set ===")

test_dataset = EntityFramingDataset(test_df, tokenizer, label2id, labeled=False)
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)
pred_labels = [id2label[i] for i in preds]

# Attach predictions
test_df["predicted_label"] = pred_labels

# Save results in the same format as original annotations
os.makedirs(TEST_DATA_RESULTS, exist_ok=True)
output_file = os.path.join(TEST_DATA_RESULTS, "subtask1_predictions.tsv")

with open(output_file, "w", encoding="utf-8") as f:
    for _, row in test_df.iterrows():
        # write: doc_id \t mention \t start \t end \t main_category
        f.write(f"{row['doc_id']}\t{row['mention']}\t{row['orig_start']}\t{row['orig_end']}\t{row['predicted_label']}\n")

print(f"\n✅ Predictions saved to: {output_file}")
test_df.head()

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import os

print("\n=== Multilingual Evaluation (Updated for Doc/Paragraph/Sentence Modes) ===")

def evaluate_predictions_per_language(languages, label2id):

    all_eval_dfs = []
    language_scores = []

    for lang in languages:
        print(f"\n🌍 Evaluating language: {lang}")

        # Try per-language folder first
        results_dir = os.path.join(TEST_DATA_RESULTS, lang)
        predictions_file = os.path.join(results_dir, "subtask1_predictions.tsv")

        # Fallback to global predictions file
        if not os.path.exists(predictions_file):
            predictions_file = os.path.join(TEST_DATA_RESULTS, "subtask1_predictions.tsv")

        test_ann_path = os.path.join(TEST_DATA_PARENT, lang, "subtask-1-annotations.txt")
        if not os.path.exists(test_ann_path):
            print(f"⚠️ Missing ground truth for {lang}, skipping.")
            continue

        # ------------------------------------------------------------
        # Load predictions (UPDATED FORMAT)
        # ------------------------------------------------------------
        predictions_df = pd.read_csv(
            predictions_file,
            sep="\t",
            names=["doc_id", "mention", "orig_start", "orig_end", "predicted_label"]
        )

        # ------------------------------------------------------------
        # Load ground truth
        # ------------------------------------------------------------
        ground_truth_data = []
        with open(test_ann_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    doc_id, mention, start, end, *labels = parts
                    ground_truth_data.append({
                        "doc_id": doc_id,
                        "mention": mention,
                        "orig_start": int(start),
                        "orig_end": int(end),
                        "true_label": labels[0]
                    })

        ground_truth_df = pd.DataFrame(ground_truth_data)

        # ------------------------------------------------------------
        # Convert types
        # ------------------------------------------------------------
        predictions_df[["orig_start", "orig_end"]] = predictions_df[["orig_start", "orig_end"]].astype(int)
        ground_truth_df[["orig_start", "orig_end"]] = ground_truth_df[["orig_start", "orig_end"]].astype(int)

        # ------------------------------------------------------------
        # Merge ON GLOBAL OFFSETS (critical fix)
        # ------------------------------------------------------------
        eval_df = predictions_df.merge(
            ground_truth_df,
            on=["doc_id", "mention", "orig_start", "orig_end"],
            how="inner"
        )

        if eval_df.empty:
            print(f"⚠️ No matching examples for {lang}. This usually means your loader changed start/end but orig_start/orig_end were not kept correctly.")
            continue

        # ------------------------------------------------------------
        # Convert labels to IDs
        # ------------------------------------------------------------
        y_true = [label2id[label] for label in eval_df["true_label"]]
        y_pred = [label2id[label] for label in eval_df["predicted_label"]]

        # ------------------------------------------------------------
        # Metrics
        # ------------------------------------------------------------
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="micro", zero_division=0
        )

        language_scores.append({
            "Language": lang,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Samples": len(eval_df)
        })
        all_eval_dfs.append(eval_df)

        print(f"→ {lang} | Acc: {acc:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

    # ------------------------------------------------------------
    # Combine and compute overall metrics
    # ------------------------------------------------------------
    if not all_eval_dfs:
        print("⚠️ No evaluation data.")
        return None, None

    combined_eval = pd.concat(all_eval_dfs, ignore_index=True)

    combined_y_true = [label2id[label] for label in combined_eval["true_label"]]
    combined_y_pred = [label2id[label] for label in combined_eval["predicted_label"]]

    overall_acc = accuracy_score(combined_y_true, combined_y_pred)
    overall_prec, overall_rec, overall_f1, _ = precision_recall_fscore_support(
        combined_y_true, combined_y_pred, average="micro", zero_division=0
    )

    print("\n=== 🌐 Overall Multilingual Evaluation ===")
    print(f"Accuracy : {overall_acc:.4f}")
    print(f"Precision: {overall_prec:.4f}")
    print(f"Recall   : {overall_rec:.4f}")
    print(f"F1       : {overall_f1:.4f}")

    language_report = pd.DataFrame(language_scores).sort_values("F1", ascending=False)

    print("\n=== Per-Language Summary ===")
    print(language_report.to_string(index=False))

    return combined_eval, language_report


# Run multilingual evaluation
combined_eval_df, language_report_df = evaluate_predictions_per_language(available_languages, label2id)

# Optional: per-class metrics
if combined_eval_df is not None:
    y_true = [label2id[label] for label in combined_eval_df["true_label"]]
    y_pred = [label2id[label] for label in combined_eval_df["predicted_label"]]

    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    performance_df = pd.DataFrame({
        "Category": list(label2id.keys()),
        "Precision": class_precision,
        "Recall": class_recall,
        "F1-Score": class_f1,
        "Support": class_support
    })

    print("\n=== Per-Category Performance (Multilingual Combined) ===")
    print(performance_df.sort_values("F1-Score", ascending=False))


In [ ]:
# =====================================
# 📊 DATA ANALYSIS & VISUALIZATION
# =====================================
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Basic Info ----------
print("\n=== Dataset Overview ===")
print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")
print(f"Unique labels: {train_df['label'].nunique()}")
print(f"Languages detected earlier: {available_languages}")

# ---------- Label Distribution ----------
plt.figure(figsize=(10,5))
train_df['label'].value_counts().plot(kind='bar')
plt.title("Label distribution in Training Set")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------- Train vs Validation Label Distribution ----------
label_counts_train = train_df['label'].value_counts(normalize=True)
label_counts_val = val_df['label'].value_counts(normalize=True)

compare_df = pd.DataFrame({
    'Train': label_counts_train,
    'Validation': label_counts_val
}).fillna(0)

compare_df.plot(kind='bar', figsize=(10,5))
plt.title("Relative Label Distribution: Train vs Validation")
plt.ylabel("Proportion")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------- Paragraph (context) Length Distribution ----------
train_df['text_length'] = train_df['text'].apply(lambda x: len(x))
val_df['text_length'] = val_df['text'].apply(lambda x: len(x))

plt.figure(figsize=(8,5))
sns.histplot(train_df['text_length'], color='blue', label='Train', kde=True)
sns.histplot(val_df['text_length'], color='orange', label='Validation', kde=True)
plt.title("Context (Paragraph) Length Distribution")
plt.xlabel("Character length of context window")
plt.legend()
plt.tight_layout()
plt.show()

# ---------- Mentions per Document ----------
mentions_per_doc = train_df.groupby("doc_id").size().sort_values(ascending=False)
plt.figure(figsize=(8,4))
sns.histplot(mentions_per_doc, bins=30, kde=False)
plt.title("Distribution of Mentions per Document (Training Set)")
plt.xlabel("Mentions per Document")
plt.ylabel("Count of Documents")
plt.tight_layout()
plt.show()

# ---------- Language-Level Label Distribution ----------
# You already have multilingual data combined, so infer language if available in doc_id path
def detect_language(doc_id):
    # simple heuristic: some doc_ids contain language code (e.g., 'EN_', 'BG_', etc.)
    for lang in available_languages:
        if doc_id.startswith(lang) or f"/{lang}/" in doc_id:
            return lang
    return "Unknown"

if 'language' not in train_df.columns:
    train_df['language'] = train_df['doc_id'].apply(detect_language)

lang_label_counts = train_df.groupby(['language', 'label']).size().reset_index(name='count')
pivot = lang_label_counts.pivot(index='label', columns='language', values='count').fillna(0)

pivot.plot(kind='bar', figsize=(10,5))
plt.title("Label Distribution per Language (Training Set)")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------- Correlation: Context length vs Label frequency ----------
plt.figure(figsize=(8,5))
sns.boxplot(data=train_df, x='label', y='text_length')
plt.title("Text Length per Label (Training Set)")
plt.xlabel("Label")
plt.ylabel("Context length (characters)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------- Average Text Length per Label ----------
avg_len_per_label = train_df.groupby("label")["text_length"].mean().sort_values(ascending=False)

plt.figure(figsize=(10,5))
avg_len_per_label.plot(kind='bar', color='teal')
plt.title("Average Context Length per Label (Training Set)")
plt.ylabel("Average character length")
plt.xlabel("Label")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------- Mention Position vs Context Length ----------
plt.figure(figsize=(8,5))
sns.scatterplot(data=train_df.sample(min(2000, len(train_df))), x='start', y='text_length', alpha=0.6)
plt.title("Mention Start Position vs Context Length (sampled)")
plt.xlabel("Mention Start Character Index")
plt.ylabel("Context Length (characters)")
plt.tight_layout()
plt.show()

# ---------- Mentions per Language ----------
lang_counts = train_df['language'].value_counts().sort_values(ascending=False)

plt.figure(figsize=(6,4))
lang_counts.plot(kind='pie', autopct='%1.1f%%', startangle=140)
plt.title("Share of Mentions per Language (Training Set)")
plt.ylabel("")
plt.tight_layout()
plt.show()

# ---------- Top 10 Most Frequent Mentions ----------
top_mentions = train_df['mention'].value_counts().head(10)

plt.figure(figsize=(8,4))
sns.barplot(x=top_mentions.values, y=top_mentions.index, palette="viridis")
plt.title("Top 10 Most Frequent Mentions (Training Set)")
plt.xlabel("Count")
plt.ylabel("Mention")
plt.tight_layout()
plt.show()


# ---------- Save basic statistics ----------
summary_stats = {
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "unique_labels": train_df['label'].nunique(),
    "avg_text_length": train_df['text_length'].mean(),
    "max_text_length": train_df['text_length'].max(),
    "min_text_length": train_df['text_length'].min()
}
print("\n=== Summary Statistics ===")
for k, v in summary_stats.items():
    print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")
